**Our research question**

Can a team-level model determine whether the current starter at a position is worth keeping relative to positional peers and their cost?

In [1]:
# Our imports!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nflreadpy as nfl

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
)

In [2]:
# Play-by-play data from nflverse (via nflreadpy), 2015-2024
seasons = list(range(2015, 2025))

columns_we_need = [
    "season", "game_id", "desc", "down", "play_type",
    "fourth_down_converted", "fourth_down_failed",
    "ydstogo", "yardline_100", "qtr", "game_seconds_remaining",
    "score_differential", "posteam_timeouts_remaining", "wp", "shotgun",
]
pbp = nfl.load_pbp(seasons).select(columns_we_need).to_pandas()

print(f"Rows in the play-by-play data: {pbp.shape[0]:,}")
print(f"Columns we kept: {pbp.shape[1]}")

Rows in the play-by-play data: 483,605
Columns we kept: 15


In [3]:
pbp.head(3)

,season,game_id,desc,down,play_type,fourth_down_converted,fourth_down_failed,ydstogo,yardline_100,qtr,game_seconds_remaining,score_differential,posteam_timeouts_remaining,wp,shotgun
0,2015,2015_01_BAL_DEN,GAME,NaN,NaN,NaN,NaN,0.0,NaN,1.0,3600.0,NaN,NaN,0.422024,0.0
1,2015,2015_01_BAL_DEN,8-B.McManus kicks 65 yards from DEN 35 to end ...,NaN,kickoff,0.0,0.0,0.0,35.0,1.0,3600.0,0.0,3.0,0.422024,0.0
2,2015,2015_01_BAL_DEN,(15:00) 5-J.Flacco pass short left to 44-K.Jus...,1.0,pass,0.0,0.0,10.0,80.0,1.0,3600.0,0.0,3.0,0.422024,0.0


# Getting salaries

- Get NFL player salary data for roughly the last 10 years so we can see whether a player’s performance is over- or undervalued relative to their pay, compared with others at the same position.
- NFL verse already has a contract feature from OverTheCap.com so we'll use that
- However each row is one player's contract, not contracts throughout the years. But the year by year salary is nested inside the "season_history" column that holds a list of dicts, one per column.
- We will chose between cap_number (the team’s cap hit that year) or cap_percent (cap hit as a share of that year’s salary cap). The percent version makes different years comparable, since the cap grows.
- Because the salaries are nested in season_history we must flatten out the data. 
    - This is done by:
        - explode gives each season its own row.
        - pd.json_normalize turns each dict into columns.
        - concat reattaches the player name and position

In [4]:
contracts = nfl.load_contracts().to_pandas()

print(f"Rows: {contracts.shape[0]:,}")
print(contracts.columns.tolist())
contracts.head()

Rows: 53,507
['player', 'position', 'team', 'is_active', 'year_signed', 'years', 'value', 'apy', 'guaranteed', 'apy_cap_pct', 'inflated_value', 'inflated_apy', 'inflated_guaranteed', 'player_page', 'otc_id', 'gsis_id', 'height', 'weight', 'college', 'draft_year', 'draft_round', 'draft_overall', 'draft_team', 'date_of_birth', 'season_history', 'contract_history']


,player,position,team,is_active,year_signed,years,value,apy,guaranteed,apy_cap_pct,...,height,weight,college,draft_year,draft_round,draft_overall,draft_team,date_of_birth,season_history,contract_history
0,Joe Burrow,QB,Bengals,True,2023,5.0,275.000,55.000000,146.510,0.245,...,"6'4""",215,LSU,2020.0,1.0,1.0,Bengals,NaN,"[{'year': '2020', 'team': 'Bengals', 'base_sal...","[{'team': 'Bengals', 'contract_type': 'Drafted..."
1,Aaron Rodgers,QB,NYJ/GB,False,2022,3.0,150.815,50.271667,101.415,0.241,...,"6'2""",225,California,2005.0,1.0,24.0,Packers,NaN,"[{'year': '2005', 'team': 'Packers', 'base_sal...","[{'team': 'Packers', 'contract_type': 'Drafted..."
2,Josh Allen,QB,Bills,False,2021,6.0,258.000,43.000000,100.000,0.236,...,"6'5""",233,Wyoming,2018.0,1.0,7.0,Bills,NaN,"[{'year': '2018', 'team': 'Bills', 'base_salar...","[{'team': 'Bills', 'contract_type': 'Drafted',..."
3,Russell Wilson,QB,Broncos,False,2022,5.0,245.000,49.000000,124.000,0.235,...,"5'11""",206,Wisconsin,2012.0,3.0,75.0,Seahawks,"November 29, 1988","[{'year': '2012', 'team': 'Seahawks', 'base_sa...","[{'team': 'Seahawks', 'contract_type': 'Drafte..."
4,Dak Prescott,QB,Cowboys,True,2024,4.0,240.000,60.000000,129.000,0.235,...,"6'2""",226,Mississippi State,2016.0,4.0,135.0,Cowboys,NaN,"[{'year': '2016', 'team': 'Cowboys', 'base_sal...","[{'team': 'Cowboys', 'contract_type': 'Drafted..."


In [ ]:
# See what's inside the nested column for one player
burrow = contracts[contracts["player"] == "Joe Burrow"].iloc[0]
print(burrow["season_history"][0])   

{'year': '2020', 'team': 'Bengals', 'base_salary': 0.61, 'prorated_bonus': 5.970025, 'option_bonus': 0.0, 'roster_bonus': 0.0, 'guaranteed_salary': 0.61, 'cap_number': 6.580025, 'cap_percent': 0.032, 'cash_paid': 24.4901, 'workout_bonus': None, 'per_game_roster_bonus': None, 'other_bonus': None}


Looks like salary is in millions. We should use cap_number or cap_percent to see what a team spent on a given player. 

In [ ]:
# Right now, the "season_history" column is just a list of dictionaries. We wanna make them into rows instead of a list so we can analyze them easier. 
# We use the pandas explode() function to do that. It will take each dictionary in the list and make it into its own row, while keeping the other columns the same.
step1 = contracts[["player", "position", "season_history"]].explode("season_history")
print(step1.shape)

# If a player has no history, that cell is NaN, so drop those
step1 = step1.dropna(subset=["season_history"])
step1.head()

(323319, 3)


,player,position,season_history
0,Joe Burrow,QB,"{'year': '2020', 'team': 'Bengals', 'base_sala..."
0,Joe Burrow,QB,"{'year': '2021', 'team': 'Bengals', 'base_sala..."
0,Joe Burrow,QB,"{'year': '2022', 'team': 'Bengals', 'base_sala..."
0,Joe Burrow,QB,"{'year': '2023', 'team': 'Bengals', 'base_sala..."
0,Joe Burrow,QB,"{'year': '2024', 'team': 'Bengals', 'base_sala..."


In [9]:
# Next we use pandas json_normalize() to turn our dicts into columns.
detail = pd.json_normalize(step1["season_history"].tolist())
print(detail.columns.tolist())
detail.head()

['year', 'team', 'base_salary', 'prorated_bonus', 'option_bonus', 'roster_bonus', 'guaranteed_salary', 'cap_number', 'cap_percent', 'cash_paid', 'workout_bonus', 'per_game_roster_bonus', 'other_bonus']


,year,team,base_salary,prorated_bonus,option_bonus,roster_bonus,guaranteed_salary,cap_number,cap_percent,cash_paid,workout_bonus,per_game_roster_bonus,other_bonus
0,2020,Bengals,0.610000,5.970025,0.0,0.000000,0.610000,6.580025,0.032,24.490100,NaN,NaN,NaN
1,2021,Bengals,0.780000,5.970025,0.0,1.475006,2.255006,8.225035,0.042,2.255006,NaN,NaN,NaN
2,2022,Bengals,0.895000,5.970025,0.0,3.005012,3.900012,9.870037,0.046,3.900012,NaN,NaN,NaN
3,2023,Bengals,1.010000,13.970025,0.0,4.535018,5.545018,19.515043,0.086,45.545018,NaN,NaN,NaN
4,2024,Bengals,10.714982,7.836130,11.0,0.000000,10.714982,29.551112,0.112,65.714982,NaN,NaN,NaN


In [ ]:
# Now we can combine the player and position columns with the new detail columns into one dataframe.
seasons_df = pd.concat(
    [step1[["player", "position"]].reset_index(drop=True), detail],
    axis=1,
)
seasons_df.head()

,player,position,year,team,base_salary,prorated_bonus,option_bonus,roster_bonus,guaranteed_salary,cap_number,cap_percent,cash_paid,workout_bonus,per_game_roster_bonus,other_bonus
0,Joe Burrow,QB,2020,Bengals,0.610000,5.970025,0.0,0.000000,0.610000,6.580025,0.032,24.490100,NaN,NaN,NaN
1,Joe Burrow,QB,2021,Bengals,0.780000,5.970025,0.0,1.475006,2.255006,8.225035,0.042,2.255006,NaN,NaN,NaN
2,Joe Burrow,QB,2022,Bengals,0.895000,5.970025,0.0,3.005012,3.900012,9.870037,0.046,3.900012,NaN,NaN,NaN
3,Joe Burrow,QB,2023,Bengals,1.010000,13.970025,0.0,4.535018,5.545018,19.515043,0.086,45.545018,NaN,NaN,NaN
4,Joe Burrow,QB,2024,Bengals,10.714982,7.836130,11.0,0.000000,10.714982,29.551112,0.112,65.714982,NaN,NaN,NaN


In [14]:
seasons_df.shape


(323275, 15)

In [1]:
print("test git commit")

test git commit


In [1]:
print("hello world")

hello world
